<div style="border-left: 4px solid #f39c12; padding-left: 15px; margin-bottom: 20px;">
  <h1 style="margin-top: 0;">☀️ 02. Exploratory Data Analysis: Renewables, Weather & Market Signals</h1>
  <p style="font-size: 1.1em; color: #888;">Connecting the factory floor to the outside world by tracking solar efficiency and European energy prices.</p>
</div>

### 🎯 Main Goals & Business Value
This notebook bridges the gap between our isolated factory and the massive energy grid. We are looking for the absolute best times to run heavy manufacturing:

*   **🌡️ Weather vs. Solar:** How much does a hot day actually reduce our solar panel efficiency? We need to know exactly how much green energy we can count on at any given hour.
*   **💶 Chasing Cheap Energy:** How much do daily electricity prices fluctuate? We want to map out these changes to find those golden windows where grid power is extremely cheap (or even negative).

### 🛠️ Core Tools & Approach
*   **API Integration:** Custom Python code to automatically pull live data from the energy market (`ENTSO-E/SMARD`) and weather databases (`DWD`).
*   **Data Engineering:** `Polars` is used for blazing-fast data processing. We need it to perfectly synchronize hourly market prices with high-speed machine telemetry, as this real-world industrial dataset is simply too heavy for Pandas.
*   **Domain Focus:** Applying sustainable architecture principles directly to reducing industrial carbon emissions.

---

### 🗄️ Data Structure
This dataset (`IPE_PV_final_features.parquet`) mathematically links our local factory hardware to the real-world power grid.

| Domain | Feature Examples | Purpose |
| :--- | :--- | :--- |
| ⚡ **Solar Power** | `DC_Power`, `inverter_efficiency` | Raw solar energy gathered and how well we convert it into usable factory power. |
| ☁️ **Climate Data** | `Air_Temperature_C` | Real weather data showing when the solar panels might overheat and lose efficiency. |
| 💶 **Market Prices** | `DayAhead_Price` | The actual hourly cost of grid electricity, which drives our cost-saving plans. |

> 
**Next Step Alignment:** The market price changes we uncover here will become the cheat sheet for our optimization program, allowing it to schedule heavy machine operations precisely when energy is cheapest and greenest.

### 🛠️ 2. Environment Setup for EDA
Importing the core tools required for numerical math, fast data manipulation, and visual charts. System warnings are hidden to keep the notebook outputs clean and easy to read.

In [27]:
# Standard Library
import datetime
import warnings

# Data Manipulation
import polars as pl

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Notebook Configuration
%matplotlib inline
warnings.filterwarnings('ignore')

# Set Polars config for clean, readable terminal outputs
pl.Config.set_tbl_rows(8)
pl.Config.set_fmt_str_lengths(50)

polars.config.Config

### 📥 3. Read the Dataset
Loading our final, cleaned external dataset (`IPE_PV_final_features.parquet`) into memory. We use **Polars** here to instantly load the heavy feature matrix, bypassing the long wait times typical of standard tools and ensuring our pipeline remains fast and production-ready.

In [28]:
# Load the dataset into memory
data_path = "../../data/processed/features/IPE_PV_final_features.parquet"

try:
    df_env = pl.read_parquet(data_path)
    
    print("✅ Dataset loaded successfully.")
    print("-" * 60)
    print(f"• Total Records:  {df_env.height:,} rows")
    print(f"• Total Features: {df_env.width} columns")
    print(f"• Date Range:     {df_env['WsDateTime'].min().date()} to {df_env['WsDateTime'].max().date()}")
    print(f"• Memory Footprint: {df_env.estimated_size('mb'):.2f} MB")
    print("-" * 60)
    
    # Display a clean snapshot of the first few rows
    display(df_env.head(3))
    
except FileNotFoundError:
    print(f"❌ Error: Could not find the file at {data_path}. Please check the folder path.")

✅ Dataset loaded successfully.
------------------------------------------------------------
• Total Records:  1,710,720 rows
• Total Features: 20 columns
• Date Range:     2024-01-01 to 2024-04-08
• Memory Footprint: 226.77 MB
------------------------------------------------------------


WsDateTime,AC_ActivePower,DailyYield,DC_Current_1,DC_Current_2,DC_Power_1,DC_Power_2,DC_Voltage_1,DC_Voltage_2,GridFreq,Systemtime,TotalYield,hour_of_day,day_of_week,month_of_year,AC_ActivePower_roll_mean_15m,AC_ActivePower_roll_std_15m,AC_ActivePower_lag_1m,Air_Temperature_C,DayAhead_Price_EUR_MWh
datetime[ms],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,f64,f64,f64,f64,f64
2024-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.7041e9,0.0,0,1,1,9.8686e-18,0.0,0.0,7.0,0.01
2024-01-01 00:00:05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.7041e9,0.0,0,1,1,9.8686e-18,0.0,0.0,7.0,0.01
2024-01-01 00:00:10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.7041e9,0.0,0,1,1,9.8686e-18,0.0,0.0,7.0,0.01


### 🧹 4. Final Health Check: Making Sure the Numbers Match Reality
While I already cleaned and structured this dataset upstream in the ELT pipeline, it is best practice to run a final validation before extracting business insights. 

In industrial data, "clean" means more than just checking for nulls. We need to make sure the numbers actually make sense in the real world. This high-speed diagnostic double-checks that we have:
* **No missing data or duplicate timestamps.**
* **No massive dropouts** in our yearly timeline.
* **No impossible physics** (like solar panels generating negative power).

In [29]:
print("=" * 70)
print("🧹 COMPREHENSIVE PIPELINE & INTEGRITY DIAGNOSTIC")
print("=" * 70)

# 1. Structural Integrity (Nulls & Duplicates)
# High-speed vectorized null check using unpivot
null_counts = df_env.select(pl.all().null_count())
missing_data = (
    null_counts
    .unpivot(variable_name="Feature", value_name="Missing_Count")
    .filter(pl.col("Missing_Count") > 0)
)
missing_passed = missing_data.height == 0

duplicate_count = df_env.filter(pl.col("WsDateTime").is_duplicated()).height
dup_passed = duplicate_count == 0

print(f"{'✅' if missing_passed else '⚠️'} Null Check:      { 'Zero missing values detected.' if missing_passed else f'{missing_data.height} columns have missing values.' }")
print(f"{'✅' if dup_passed else '⚠️'} Duplicate Check: { 'Perfectly unique temporal grid.' if dup_passed else f'{duplicate_count:,} duplicate timestamps found.' }")

# 2. Temporal Continuity & Coverage
start_time = df_env.select(pl.col("WsDateTime").min()).item()
end_time = df_env.select(pl.col("WsDateTime").max()).item()
max_gap_s = df_env.select(pl.col("WsDateTime").diff().dt.total_seconds().max()).item()
max_gap_hrs = (max_gap_s / 3600.0) if max_gap_s is not None else 0

print(f"✅ Time Coverage:   {start_time.date()} to {end_time.date()} ({(end_time - start_time).days} days)")
if max_gap_hrs > 24:
    print(f"⚠️ Gap Warning:     Longest telemetry dropout is {max_gap_hrs:.1f} hours.")
else:
    print(f"✅ Continuity Check:No major multi-day sensor dropouts (Max gap: {max_gap_hrs:.1f} hours).")

# 3. Physical Boundary Checks (No negative generation/loads)
# We specifically do NOT check the market price column here, as prices CAN go negative!
physical_keywords = ["power", "current", "voltage", "yield"]
physical_cols = [
    c for c in df_env.columns 
    if any(kw in c.lower() for kw in physical_keywords) 
    and df_env[c].dtype in [pl.Float32, pl.Float64, pl.Int32, pl.Int64]
]

# Vectorized min calculation for all physical columns simultaneously
min_exprs = [pl.col(c).min().alias(c) for c in physical_cols]
min_values = df_env.select(min_exprs).row(0)

negative_issues = [physical_cols[i] for i, val in enumerate(min_values) if val is not None and val < 0]

if not negative_issues:
    print(f"✅ Physics Check:   Passed. All {len(physical_cols)} solar metrics show valid positive ranges.")
else:
    print(f"⚠️ Physics Warning: Negative values detected in {len(negative_issues)} sensors: {negative_issues[:3]}...")

# 4. Final Pipeline Status
print("-" * 70)
if missing_passed and dup_passed and not negative_issues:
    print("🚀 PIPELINE STATUS: PRODUCTION-READY")
    print("The dataset is structurally sound, physically logical, and ready for modeling.")
else:
    print("⚠️ PIPELINE STATUS: REQUIRES CLEANING")
    print("Please address the warnings above before proceeding to modeling.")
print("-" * 70)

# 5. Display statistical summary for key sensors to visually verify sane ranges
print("\n📊 Key Sensor Summary Statistics (Sample):")
display(df_env.select(physical_cols[:5]).describe())

🧹 COMPREHENSIVE PIPELINE & INTEGRITY DIAGNOSTIC
✅ Null Check:      Zero missing values detected.
✅ Duplicate Check: Perfectly unique temporal grid.
✅ Time Coverage:   2024-01-01 to 2024-04-08 (98 days)
✅ Continuity Check:No major multi-day sensor dropouts (Max gap: 0.0 hours).
✅ Physics Check:   Passed. All 12 solar metrics show valid positive ranges.
----------------------------------------------------------------------
🚀 PIPELINE STATUS: PRODUCTION-READY
The dataset is structurally sound, physically logical, and ready for modeling.
----------------------------------------------------------------------

📊 Key Sensor Summary Statistics (Sample):


statistic,AC_ActivePower,DailyYield,DC_Current_1,DC_Current_2,DC_Power_1
str,f64,f64,f64,f64,f64
"""count""",1.71072e6,1.71072e6,1.71072e6,1.71072e6,1.71072e6
"""null_count""",0.0,0.0,0.0,0.0,0.0
"""mean""",0.350175,766.719499,1.3584e6,90.774251,0.352897
"""std""",1.180657,3367.523142,3.8633e6,168.006908,1.188702
…,…,…,…,…,…
"""25%""",0.0,0.0,0.0,0.0,0.0
"""50%""",0.0,0.0,0.0,0.0,0.0
"""75%""",0.0,0.0,0.0,0.0,0.0
"""max""",10.57,34390.0,1.2933897e7,481.28,10.37


> **⏱️ Note on Temporal Coverage (98 Days)**
> 
> You may notice this dataset covers 98 days, whereas the factory machine telemetry in Notebook **01_machine_telemetry_eda** covered a full year. This is a deliberate feature of our robust ETL pipeline:
> 
> * **Weather & Prices:** Begin on January 1, 2024.
> * **Solar Telemetry:** Ends on April 8, 2024.
> 
> To guarantee our ML models never train on missing or incomplete rows, the pipeline automatically filtered for the exact window where all three data streams perfectly intersect. Thanks to our ultra-fast 5-second sensor resolution, this 98-day window still provides over 1.7 million rows of flawless, production-ready data to map our climate impacts.

### 🌡️ 5. Climate Resilience: Why Heat Hurts Solar Power
My background in sustainable architecture taught me a simple rule: solar panels need light, but they hate the heat. When the outside air gets too hot, the panels overheat and actually generate less electricity. 

To see exactly how much power our factory loses on a warm day, we are going to group millions of 5-second sensor readings by the outside temperature. By focusing just on daylight hours, we can find the exact temperature where our green energy starts to drop off.

In [30]:
# 🚀 Thermal Aggregation

# 1. Filter for active daylight generation (avoiding nighttime zeros)
daylight_pv = df_env.filter(pl.col("AC_ActivePower") > 5.0)

# 2. Vectorized grouping: Round temps to nearest degree and calculate mean power
temp_efficiency = (
    daylight_pv
    .with_columns(pl.col("Air_Temperature_C").round(0).cast(pl.Int32).alias("Temp_Bucket"))
    .group_by("Temp_Bucket")
    .agg(pl.col("AC_ActivePower").mean().alias("Mean_Power"))
    .sort("Temp_Bucket")
).to_pandas()

# 3. Visualize the thermal curve
fig = px.scatter(
    temp_efficiency,
    x="Temp_Bucket",
    y="Mean_Power",
    title="PV Yield vs. Ambient Temperature (Thermal Degradation Curve)",
    labels={
        "Temp_Bucket": "Ambient Air Temperature (°C)", 
        "Mean_Power": "Average Active Power Output (kW)"
    },
    template="plotly_white"
)

fig.update_traces(marker=dict(size=12, color="#e67e22", opacity=0.8, line=dict(width=1, color="DarkSlateGrey")))
fig.update_layout(hovermode="x unified")
fig.show()

> **💡 The Thermal Breaking Point**
> 
> The data confirms our architectural hypothesis: solar efficiency peaks at a cool 10°C, and as ambient temperatures continue to climb toward 20°C, the panels overheat and lose nearly 1.5 kW of average power output.

### 📉 6. Market Dynamics: The Midday Price Drop
Now we connect the weather directly to the wallet. 

Germany has so much solar infrastructure that sunny days actually flood the grid with excess power. This creates a famous market quirk called the "cannibalization effect" (or the Duck Curve)—where this massive midday oversupply drives electricity prices straight down.

To visualize this, I will plot our average solar yield against the day-ahead grid prices. If the math checks out, we should see electricity prices dip exactly when the sun hits its peak.

In [31]:
# 🚀 Daily Market Profile

# 1. Aggregate average solar yield and grid price by hour of the day
hourly_market = (
    df_env
    .group_by("hour_of_day")
    .agg([
        pl.col("AC_ActivePower").mean().alias("avg_solar"),
        pl.col("DayAhead_Price_EUR_MWh").mean().alias("avg_price")
    ])
    .sort("hour_of_day")
).to_pandas()

# 2. Build a dual-axis visualization
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add Solar curve (Primary Y-Axis)
fig.add_trace(
    go.Scatter(
        x=hourly_market["hour_of_day"], 
        y=hourly_market["avg_solar"],
        name="Avg Solar Yield (kW)",
        line=dict(color="#f39c12", width=3, shape="spline"),
        fill="tozeroy",
        opacity=0.6
    ),
    secondary_y=False,
)

# Add Price curve (Secondary Y-Axis)
fig.add_trace(
    go.Scatter(
        x=hourly_market["hour_of_day"], 
        y=hourly_market["avg_price"],
        name="Avg Grid Price (€/MWh)",
        line=dict(color="#2980b9", width=3, dash="dot", shape="spline")
    ),
    secondary_y=True,
)

# 3. Format professional layout
fig.update_layout(
    title="Market Dynamics: The Daily Solar Cannibalization Effect (98-Day Avg)",
    xaxis=dict(title="Hour of Day (0-23)", tickmode="linear", tick0=0, dtick=1),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_yaxes(title_text="Average Solar Yield (kW)", secondary_y=False, color="#f39c12")
fig.update_yaxes(title_text="Day-Ahead Price (€/MWh)", secondary_y=True, color="#2980b9")

fig.show()

> **💡 The Midday Discount (The "Duck Curve")**
> 
> Because millions of solar panels across Germany flood the grid simultaneously, day-ahead electricity prices consistently crash to their lowest point precisely at 12:00 PM—creating the perfect scheduling window for our heavy factory machines.

### 🏁 Executive Summary: Goals Achieved
I opened this notebook with two primary business questions. By merging our high-speed factory telemetry with external climate and market APIs, we now have definitive, data-backed answers:

* **🌡️ Weather vs. Solar (Thermal Degradation):** I empirically proved that our solar panels hit their maximum efficiency at a cool 10°C. As ambient temperatures climb toward 20°C, the photovoltaic cells suffer from thermal degradation, losing nearly 1.5 kW of average output. Our future scheduling AI must account for this, recognizing that hot summer afternoons will yield less baseline green energy.
* **💶 Chasing Cheap Energy (The Duck Curve):** I successfully mapped the daily European market lifecycle. Driven by Germany's massive renewable infrastructure, midday solar surges create a consistent oversupply that crashes grid electricity prices to their absolute lowest point right around 12:00 PM.

**The Next Step: AI Plant Optimization** 
We now have the complete blueprint. From `01_machine_telemetry_eda.ipynb`, we know the `TEC_48S` manufacturing machine is highly flexible, requiring only 2–5 hours to complete its daily batch. `02_energy_grid_weather_eda.ipynb`, we know the most profitable time to run that batch is exactly at noon. 

In the final exploratory phase (`03_plant_optimization_eda.ipynb`), I will combine these insights to simulate the financial impact, calculating exactly how much capital we can save by migrating this machine from a blind schedule to an AI-driven, price-optimized workflow.